# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [1]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [2]:
# batch_size와 epochs를 조정해보세요!
batch_size = 16
epochs = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# 데이터셋 로드
raw_datasets = load_dataset("nyu-mll/glue", "sst2")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [6]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [9]:
# 학습 함수 정의
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    train_dataset = tokenized_datasets['train']
    valid_dataset = tokenized_datasets['validation']

    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc

# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1:  39%|███▉    

KeyboardInterrupt: 

## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명
>  * BERT: Transformer encoder만 사용하며, 사전학습 시 입력 토큰의 약 15%를 [MASK]로 가린 뒤 양방향 문맥으로 원래 토큰을 맞히는 Masked Language Model(MLM) 방식으로 학습합니다. 추가로 두 문장이 이어지는지 맞히는 NSP(Next Sentence Prediction)도 보조적으로 사용됩니다. 가려진 15% 토큰에서만 학습 신호(loss)가 발생합니다.
> * ELECTRA: 작은 generator(경량 MLM)가 일부 토큰을 그럴듯한 다른 토큰으로 바꿔치기하고, 본체인 discriminator(우리가 fine-tuning하는 ELECTRA 모델)는 각 토큰이 "원본인지/치환됐는지"를 이진 분류하는 Replaced Token Detection(RTD) 방식으로 학습합니다. 입력의 모든 토큰에서 학습 신호를 얻기 때문에 BERT보다 사전학습 효율이 높습니다. (GAN과 구조는 비슷하지만 generator는 적대적 학습이 아니라 일반 MLE로 학습됩니다.)
> * 두 모델 모두 멀티헤드 셀프어텐션 기반 encoder 구조 자체는 유사하며, 차이는 사전학습 목표(objective)에 있습니다.
2. 어떤 모델이 적합한지에 대한 본인의 의견
  - 학습 속도, accuracy 등 고려
  > 같은 연산량 대비 ELECTRA는 모든 토큰에서 신호를 얻는 RTD 방식 덕분에 더 빠르게 수렴하고, GLUE 같은 벤치마크에서 동일 파라미터 규모(base) 기준 BERT보다 대체로 더 높은 정확도를 보입니다. 다만 이번 과제처럼 이미 사전학습된 모델을 SST-2에 파인튜닝하는 경우, 두 모델 모두 encoder 구조와 파라미터 수(둘 다 약 1.1억 개)가 비슷하므로 파인튜닝 자체의 학습 속도(epoch당 시간)는 큰 차이가 없을 가능성이 높습니다. 차이는 주로 최종 validation accuracy에서 ELECTRA가 약간 더 우세하게 나타나는 경우가 많습니다.
→ 결론적으로 정확도를 우선시한다면 ELECTRA가 더 적합하다고 볼 수 있고, BERT는 더 널리 쓰이고 자료/생태계가 풍부해 안정성·범용성 면에서 강점이 있습니다.
